# Aircraft Turnaround Analytics — US Domestic Network (2024)

Reconstructs **every aircraft turnaround** in the 2024 US Bureau of Transportation Statistics (BTS) on-time dataset by linking consecutive flights of the same airframe (tail number), then asks three questions:

1. **How are turnaround times distributed, and how predictable are they** across airlines and airports?
2. **Can we predict which turnarounds will generate a departure delay** — and how much does each layer of operational information (calendar → published schedule → live inbound status) add?
3. **Where and when is the network most fragile** — by hour, weekday, season and airport?

| Stage | Methods |
|---|---|
| 1. Setup | Colab / local bootstrap |
| 2. Data acquisition & turnaround reconstruction | BTS download (cached as parquet), absolute-timestamp linking of aircraft rotations |
| 3. Exploratory data analysis | Ground-time distributions, airline spread, seasonality |
| 4. Statistical distribution modeling | MLE fits, log-normal mixtures, ΔAIC / BIC, KS effect size |
| 5. Information-theoretic variability | Shannon entropy, KL divergence, Jensen–Shannon distance |
| 6. Uncertainty quantification | Aircraft-level cluster bootstrap & jackknife, design effects |
| 7. Airport segmentation | PCA + K-Means with silhouette-based model selection |
| 8. Predictive modeling | Gradient boosting on three information tiers, date-grouped validation, calibration, permutation importance |
| 9. Operational scenario analysis | Hour × weekday and month × airline risk calendars |
| 10. Network risk analysis | Interactive US risk map, turn-buffer reliability, airport seasonality |
| 11. Key results & export | Metrics JSON, result tables, figure bundle |

All helper code lives in [`src/`](../src); every figure is written to [`images/`](../images) and every table to [`outputs/`](../outputs).

## 1. Setup

Works in **Google Colab** and locally. On Colab the cell below fetches the repository (from an uploaded `aircraft-turnaround-analytics.zip` if present, otherwise by cloning from GitHub) and installs `requirements.txt`. A standard CPU runtime is enough — nothing here uses a GPU.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_NAME = "aircraft-turnaround-analytics"
REPO_URL = f"https://github.com/sm25mtech11008-beep/{REPO_NAME}.git"  # edit if you rename the repo
IN_COLAB = "google.colab" in sys.modules


def find_project_root(start: Path):
    for p in [start, *start.parents]:
        if (p / "src" / "config.py").exists():
            return p
    return None


ROOT = find_project_root(Path.cwd())
if ROOT is None and IN_COLAB:
    ROOT = Path("/content") / REPO_NAME
    if not ROOT.exists():
        uploaded_zip = Path("/content") / f"{REPO_NAME}.zip"
        if uploaded_zip.exists():
            subprocess.run(["unzip", "-q", "-o", str(uploaded_zip), "-d", "/content"], check=True)
        else:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the repository (src/config.py not found).")

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    # System libraries Chrome needs for Plotly's static image export (kaleido)
    subprocess.run("apt-get -qq update > /dev/null && apt-get -qq install -y libnss3 libatk-bridge2.0-0 "
                   "libcups2 libxcomposite1 libxdamage1 libxrandr2 libgbm1 libxkbcommon0 libpango-1.0-0 "
                   "libasound2 > /dev/null", shell=True, check=False)
print("Project root:", ROOT)

In [ ]:
import json
import time
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from sklearn.calibration import calibration_curve
from sklearn.metrics import precision_recall_curve, precision_score, recall_score, roc_curve

from src import analysis, config as cfg, data, geo, modeling, viz

warnings.filterwarnings("ignore", category=FutureWarning)
cfg.ensure_dirs()
viz.set_style()
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
RNG = np.random.default_rng(cfg.RANDOM_STATE)
RESULTS = {}  # headline numbers, written to outputs/ at the end
print("pandas", pd.__version__, "| numpy", np.__version__)

## 2. Data Acquisition & Turnaround Reconstruction

**Source:** [BTS Reporting Carrier On-Time Performance](https://www.transtats.bts.gov/), all 12 months of 2024 (~7M flights). Each monthly zip is streamed, only the 13 needed columns are kept, and the result is cached as parquet in `data/raw/` (~30 MB/month instead of ~250 MB of CSV). The finished turnaround table is cached in `data/processed/`, so re-runs skip straight to analysis. The first run takes ~5–10 min on Colab.

**Reconstruction.** Flights are sorted by tail number and scheduled departure. A departure is linked to the same aircraft's previous arrival when the aircraft is at the same airport. All times are converted to **absolute timestamps** (date + local clock + signed delay), so overnight and multi-day gaps can't wrap around midnight.

| Quantity | Definition |
|---|---|
| Planned ground time `TAT_Scheduled` | scheduled departure − previous scheduled arrival; events kept when 15–360 min (shorter = data error, longer = overnight parking) |
| Actual ground time `TAT_Actual` | actual departure − previous actual arrival |
| Ground-time overrun `TAT_Delay` | actual − planned (equal to departure delay − inbound arrival delay) |
| **Delay-generating turnaround** `TAT_Delayed` | overrun ≥ 15 min **and** outbound departure ≥ 15 min late |

The label is defined so that an aircraft that lands early and waits for its slot doesn't count as "delayed", and neither does inbound lateness that the ground crew absorbed. It flags only turnarounds where the ground phase itself produced a late departure (15 min is the FAA/DOT on-time threshold).

In [ ]:
tat, funnel, diagnostics = data.load_or_build_turnarounds(cfg.YEAR, cfg.MONTHS)

funnel_df = pd.DataFrame(funnel, columns=["Step", "Rows"])
funnel_df["Rows"] = funnel_df["Rows"].map("{:,}".format)
display(funnel_df)
print(f"{len(tat):,} turnarounds | {tat['Airline'].nunique()} airlines | "
      f"{tat['Origin'].nunique()} airports | {tat['Tail_Number'].nunique():,} aircraft")
tat.head()

**Methodology check: naive vs corrected reconstruction.** A naive pipeline computes ground time as a clock difference modulo 24 h and labels a turnaround "delayed" whenever actual ground time exceeds the plan. The table below runs that logic on the same flights to show what it gets wrong.

In [ ]:
check = pd.DataFrame(
    {
        "Naive method": [
            f"{diagnostics['naive_events']:,}",
            f"{diagnostics['naive_multi_day_aliased']:,}",
            f"{diagnostics['naive_delayed_rate']:.1%}",
        ],
        "Corrected method": [
            f"{diagnostics['events']:,}",
            "0",
            f"{diagnostics['delayed_rate']:.1%}",
        ],
    },
    index=[
        "Turnaround events",
        "…of which the aircraft actually sat > 24 h (midnight wrap-around)",
        "Share labelled 'delayed'",
    ],
)
display(check)
print(f"Ground time exceeded plan in {diagnostics['share_ground_time_over_plan']:.1%} of corrected events, "
      "mostly early arrivals waiting for their departure slot rather than ground delays.")
RESULTS.update(
    flights_raw=funnel[0][1], turnarounds=len(tat), airlines=int(tat["Airline"].nunique()),
    airports=int(tat["Origin"].nunique()), aircraft=int(tat["Tail_Number"].nunique()),
    naive_events=diagnostics["naive_events"], naive_aliased=diagnostics["naive_multi_day_aliased"],
    naive_delayed_rate=diagnostics["naive_delayed_rate"], delayed_rate=diagnostics["delayed_rate"],
)

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9.5), layout="constrained")
viz.suptitle(fig, f"Aircraft turnarounds at a glance: US domestic network, {cfg.YEAR}")

# (a) actual vs planned ground time
ax = axes[0, 0]
bins = np.arange(0, 365, 5)
ax.hist(tat["TAT_Actual"], bins=bins, color=viz.BLUE, alpha=0.85, label="Actual ground time")
ax.hist(tat["TAT_Scheduled"], bins=bins, histtype="step", color=viz.ORANGE, linewidth=1.8,
        label="Planned ground time")
med = tat["TAT_Actual"].median()
ax.axvline(med, color=viz.INK, ls="--", lw=1)
ax.text(med + 4, ax.get_ylim()[1] * 0.92, f"median {med:.0f} min", fontsize=9, color=viz.INK_2)
ax.set(title="Ground time distribution", xlabel="Minutes on the ground (shown up to 360)", ylabel="Turnarounds")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v / 1e3:,.0f}k"))
ax.legend()

# (b) overrun vs plan
ax = axes[0, 1]
ax.hist(tat["TAT_Delay"].clip(-60, 120), bins=np.arange(-60, 123, 3), color=viz.BLUE, alpha=0.85)
ax.axvline(0, color=viz.INK, lw=1)
ax.axvline(cfg.DELAY_THRESHOLD, color=viz.ORANGE, ls="--", lw=1.5, label=f"+{cfg.DELAY_THRESHOLD} min threshold")
ax.set(title="Ground time vs plan (actual − planned)", xlabel="Minutes (clipped to −60 … +120)",
       ylabel="Turnarounds")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v / 1e3:,.0f}k"))
ax.legend()

# (c) spread by airline: median + interquartile range
ax = axes[1, 0]
g = tat.groupby("Airline", observed=True)["TAT_Actual"]
air = (pd.DataFrame({"median": g.median(), "q1": g.quantile(0.25), "q3": g.quantile(0.75), "n": g.size()})
         .query("n >= 5000").sort_values("median"))
labels = air.index.astype(str)
ax.hlines(labels, air["q1"], air["q3"], color=viz.NEUTRAL, lw=7, capstyle="round", label="Interquartile range")
ax.scatter(air["median"], labels, color=viz.BLUE, s=45, zorder=3, label="Median")
ax.set(title="Ground time by airline", xlabel="Minutes on the ground")
ax.legend(loc="lower right")

# (d) seasonality
ax = axes[1, 1]
monthly = tat.groupby("Month")["TAT_Delayed"].mean().reindex(range(1, 13)) * 100
ax.plot(range(1, 13), monthly.values, marker="o", color=viz.BLUE)
ax.set_xticks(range(1, 13), cfg.MONTH_LABELS)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=1))
ax.set(title="Delay-generating turnarounds by month", ylabel="Share of turnarounds")

viz.save_fig(fig, "01_eda_overview")
plt.show()

display(tat[["TAT_Scheduled", "TAT_Actual", "TAT_Delay", "Prev_ArrDelay", "DepDelay"]]
        .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).T.round(1))
RESULTS.update(median_tat=float(med), mean_tat=float(tat["TAT_Actual"].mean()),
               median_sched_tat=float(tat["TAT_Scheduled"].median()))

## 4. Statistical Distribution Modeling

Which probability model best describes actual ground time? Candidates are fitted by maximum likelihood on a 300k random sample and ranked by **ΔAIC** (difference from the best model). With samples this large every goodness-of-fit *p*-value is ≈ 0, so the **KS statistic** (largest CDF gap) is reported as an effect size instead. Turnaround times are visibly multimodal (regional quick-turns vs mainline turns), so 2- and 3-component **log-normal mixtures** are included as candidates.

In [ ]:
fit_table, fit_pdfs = analysis.fit_distributions(tat["TAT_Actual"], sample_size=300_000)
best_dist = fit_table.iloc[0]["Distribution"]
display(fit_table[["Distribution", "Params", "LogLik", "AIC", "BIC", "dAIC", "KS_stat"]]
        .style.format({"LogLik": "{:,.0f}", "AIC": "{:,.0f}", "BIC": "{:,.0f}", "dAIC": "{:,.0f}",
                       "KS_stat": "{:.4f}"}))
print("Best fit:", best_dist)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.4), layout="constrained", gridspec_kw={"width_ratios": [1.45, 1]})
viz.suptitle(fig, "Which distribution describes turnaround time?")

ax = axes[0]
ax.hist(tat["TAT_Actual"], bins=np.arange(0, cfg.MAX_ACTUAL_TAT + 4, 4), density=True,
        color=viz.NEUTRAL, alpha=0.6, label="Observed")
grid = np.linspace(5, 360, 600)
show = [best_dist] + [d for d in ["Log-normal", "Gamma", "Weibull", "Normal"] if d != best_dist]
dAIC = fit_table.set_index("Distribution")["dAIC"]
for name, color in zip(show, viz.SERIES):
    ax.plot(grid, fit_pdfs[name](grid), color=color, lw=2.4 if name == best_dist else 1.5,
            label=f"{name} (ΔAIC = {dAIC[name]:,.0f})")
ax.set_xlim(0, 360)
ax.set(title="Fitted densities", xlabel="Actual ground time (min)", ylabel="Density")
ax.legend(fontsize=8.5)

ax = axes[1]
order = fit_table.sort_values("dAIC", ascending=False)
ax.barh(order["Distribution"], order["dAIC"] + 1,
        color=[viz.BLUE if d == best_dist else viz.NEUTRAL for d in order["Distribution"]])
ax.set_xscale("log")
ax.set_xlim(1, (order["dAIC"].max() + 1) * 40)
for i, v in enumerate(order["dAIC"]):
    ax.text(v + 1, i, f"  {v:,.0f}", va="center", fontsize=9, color=viz.INK_2)
ax.set(title="ΔAIC vs best model (lower is better)", xlabel="ΔAIC + 1 (log scale)")
ax.grid(axis="y", visible=False)

viz.save_fig(fig, "02_distribution_fit")
plt.show()
RESULTS.update(best_distribution=best_dist,
               lognormal_dAIC=float(dAIC.get("Log-normal", np.nan)),
               best_ks=float(fit_table.iloc[0]["KS_stat"]))

## 5. Information-Theoretic Variability Analysis

**Shannon entropy** of the ground-time distribution measures how *unpredictable* an airline's turnarounds are (higher = more spread over many durations). Every group is binned on the **same 5-minute grid**, which is what makes the values comparable. **KL divergence** and the symmetric, bounded **Jensen–Shannon distance** quantify how different hub and regional airports' turnaround profiles are.

In [ ]:
edges = analysis.tat_bin_edges(width=5)
entropy_airline = analysis.entropy_by_group(tat, "Airline", min_n=5_000)
entropy_airline["Airline"] = entropy_airline["Airline"].astype(str)
display(entropy_airline.round(3))

counts = tat["Origin"].value_counts()
counts = counts[counts > 0]
hubs = counts.index[:20]
regional = counts.index[max(20, len(counts) // 3):]  # the less-busy two-thirds of airports
tat_hub = tat.loc[tat["Origin"].isin(hubs), "TAT_Actual"]
tat_reg = tat.loc[tat["Origin"].isin(regional), "TAT_Actual"]
kl_hr = analysis.kl_divergence(tat_hub, tat_reg, edges)
kl_rh = analysis.kl_divergence(tat_reg, tat_hub, edges)
js = analysis.js_distance(tat_hub, tat_reg, edges)
print(f"Hubs: {len(hubs)} airports, {len(tat_hub):,} turns | Regional: {len(regional)} airports, {len(tat_reg):,} turns")
print(f"KL(hubs || regional) = {kl_hr:.3f} bits | KL(regional || hubs) = {kl_rh:.3f} bits | JS distance = {js:.3f}")
RESULTS.update(kl_hub_regional=kl_hr, kl_regional_hub=kl_rh, js_hub_regional=js,
               most_unpredictable_airline=entropy_airline.iloc[0]["Airline"],
               most_consistent_airline=entropy_airline.iloc[-1]["Airline"])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.6), layout="constrained")
viz.suptitle(fig, "How predictable are turnarounds?")

ax = axes[0]
e = entropy_airline.sort_values("Entropy_bits")
ax.barh(e["Airline"], e["Entropy_bits"],
        color=[viz.ORANGE if v == e["Entropy_bits"].max() else viz.BLUE for v in e["Entropy_bits"]])
ax.set_xlim(e["Entropy_bits"].min() - 0.3, e["Entropy_bits"].max() + 0.1)
ax.set(title="Shannon entropy of ground time by airline (higher = less predictable)", xlabel="Entropy (bits)")
ax.grid(axis="y", visible=False)

ax = axes[1]
centers = (edges[:-1] + edges[1:]) / 2
for series, name, color in [(tat_hub, "20 busiest hubs", viz.BLUE), (tat_reg, "Regional airports", viz.ORANGE)]:
    dens, _ = np.histogram(series, bins=edges, density=True)
    ax.step(centers, dens, where="mid", color=color, label=name)
ax.set_xlim(0, 300)
ax.set(title=f"Hubs vs regional airports (KL = {kl_hr:.2f} bits, JS distance = {js:.2f})",
       xlabel="Actual ground time (min)", ylabel="Density")
ax.legend()

viz.save_fig(fig, "03_entropy_kl")
plt.show()

## 6. Uncertainty Quantification

How precisely do we know each airline's mean ground time and delay rate? Turnarounds of the same aircraft are **not independent** (same fleet type, rotation and crew base), so resampling individual rows understates uncertainty. Instead, whole **aircraft are resampled** (cluster bootstrap, 2,000 replicates) and a **delete-one-aircraft jackknife** gives a second estimate. The **design effect** = (cluster SE / naive SE)² shows how much the i.i.d. assumption shrinks the error bars.

In [ ]:
unc_tat = analysis.uncertainty_table(tat, "Airline", "TAT_Actual", "Tail_Number", top_n=10, n_boot=2_000)
unc_rate = analysis.uncertainty_table(tat, "Airline", "TAT_Delayed", "Tail_Number", top_n=10, n_boot=2_000)
for t in (unc_tat, unc_rate):
    t["Airline"] = t["Airline"].astype(str)
print("Mean ground time (min) with aircraft-level 95% CI")
display(unc_tat.round(4))
print("Delay-generating rate with aircraft-level 95% CI")
display(unc_rate.round(4))
RESULTS.update(design_effect_tat_median=float(unc_tat["Design_effect"].median()),
               design_effect_rate_median=float(unc_rate["Design_effect"].median()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.6), layout="constrained")
viz.suptitle(fig, "Uncertainty in airline-level estimates")

ax = axes[0]
u = unc_tat.sort_values("Mean")
ax.errorbar(u["Mean"], u["Airline"], xerr=[u["Mean"] - u["CI_low"], u["CI_high"] - u["Mean"]],
            fmt="o", color=viz.BLUE, ecolor=viz.INK_2, elinewidth=1.5, capsize=4, markersize=6)
ax.set(title="Mean ground time with 95% aircraft-level bootstrap CI", xlabel="Minutes")

ax = axes[1]
de = (unc_tat[["Airline", "Design_effect"]].rename(columns={"Design_effect": "Ground time"})
      .merge(unc_rate[["Airline", "Design_effect"]].rename(columns={"Design_effect": "Delay rate"}), on="Airline")
      .sort_values("Ground time"))
y = np.arange(len(de))
ax.barh(y - 0.2, de["Ground time"], height=0.38, color=viz.BLUE, label="Mean ground time")
ax.barh(y + 0.2, de["Delay rate"], height=0.38, color=viz.ORANGE, label="Delay-generating rate")
ax.set_yticks(y, de["Airline"])
ax.axvline(1, color=viz.INK, lw=1, ls="--")
ax.text(1, len(de) - 0.4, " 1 = independent", fontsize=8.5, color=viz.INK_2, va="bottom")
ax.set(title="Design effect: how much i.i.d. resampling understates variance", xlabel="(cluster SE / naive SE)²")
ax.legend(loc="lower right")
ax.grid(axis="y", visible=False)

viz.save_fig(fig, "04_uncertainty")
plt.show()

## 7. Airport Segmentation (PCA + K-Means)

Each airport is described by its turnaround profile (ground time level and spread, planned buffer, delay generation, inbound / outbound delay, log volume). Features are standardised, compressed with PCA, and clustered with K-Means. The number of clusters is chosen by **silhouette score** rather than eyeballing an elbow.

In [ ]:
feats = analysis.airport_features(tat, min_events=200)
seg = analysis.segment_airports(feats, n_components=3, k_range=range(2, 9))
feats["Cluster"] = seg["labels"]
feats["PC1"], feats["PC2"] = seg["Z"][:, 0], seg["Z"][:, 1]
print(f"{len(feats)} airports | PCA variance explained: "
      + ", ".join(f"PC{i + 1} {v:.1%}" for i, v in enumerate(seg["explained"]))
      + f" | best k = {seg['best_k']} (silhouette {seg['silhouettes'][seg['best_k']]:.3f})")

cluster_profile = feats.groupby("Cluster").agg(
    Airports=("Events", "size"), Turnarounds=("Events", "sum"), Mean_TAT=("Mean_TAT", "mean"),
    Planned_TAT=("Median_Sched_TAT", "mean"), Delay_Rate=("Delay_Rate", "mean"),
    Inbound_Delay=("Mean_Inbound_Delay", "mean"), Outbound_Delay=("Mean_Outbound_Delay", "mean"))
display(cluster_profile.round(3))
print("PCA loadings")
display(seg["loadings"].round(2))
RESULTS.update(airport_clusters=int(seg["best_k"]), pca_var_3pc=float(seg["explained"].sum()))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10.5), layout="constrained")
viz.suptitle(fig, "Airport segmentation by turnaround profile")

ax = axes[0, 0]
pcs = [f"PC{i + 1}" for i in range(len(seg["explained"]))]
ax.bar(pcs, seg["explained"] * 100, color=viz.BLUE, width=0.6)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
ax.set(title="Variance explained by each component", ylabel="Variance explained")

ks = list(seg["inertias"])
ax = axes[0, 1]
ax.plot(ks, list(seg["inertias"].values()), marker="o", color=viz.BLUE)
ax.axvline(seg["best_k"], color=viz.INK_2, ls="--", lw=1)
ax.set(title="Elbow curve", xlabel="Number of clusters k", ylabel="Within-cluster inertia")

ax = axes[1, 0]
ax.plot(ks, list(seg["silhouettes"].values()), marker="o", color=viz.ORANGE)
ax.axvline(seg["best_k"], color=viz.INK_2, ls="--", lw=1, label=f"chosen k = {seg['best_k']}")
ax.set(title="Silhouette score (higher = better separated)", xlabel="Number of clusters k", ylabel="Silhouette")
ax.legend()

ax = axes[1, 1]
markers = ["o", "s", "^", "D", "v", "P", "X", "*"]
for c in sorted(feats["Cluster"].unique()):
    sub = feats[feats["Cluster"] == c]
    ax.scatter(sub["PC1"], sub["PC2"], s=28 + 220 * sub["Events"] / feats["Events"].max(),
               color=viz.SERIES[c % 8], marker=markers[c % 8], alpha=0.8, edgecolors="white",
               linewidths=0.8, label=f"Cluster {c} ({len(sub)} airports)")
for code_, r in feats.nlargest(10, "Events").iterrows():
    ax.annotate(str(code_), (r["PC1"], r["PC2"]), xytext=(5, 4), textcoords="offset points",
                fontsize=8, color=viz.INK_2)
ax.set(title="Airports in principal-component space (size = volume)",
       xlabel=f"PC1 ({seg['explained'][0]:.0%})", ylabel=f"PC2 ({seg['explained'][1]:.0%})")
ax.legend(fontsize=8)

viz.save_fig(fig, "05_airport_clusters")
plt.show()

## 8. Predictive Modeling: Which Turnarounds Will Generate a Delay?

**Target:** `TAT_Delayed` (delay-generating turnaround, defined in Section 2).

**Information tiers.** The same model is trained on three nested feature sets that match *when* a prediction is made:

| Tier | Available | Features |
|---|---|---|
| Calendar | weeks ahead | airline, airport, month, weekday, departure hour |
| Schedule | day of operation | + planned ground time, distance, leg of the aircraft's day, airport daily departures |
| Real-time | inbound aircraft has landed | + inbound arrival delay |

**Validation.** Whole **calendar days** are held out (20% of days), so a storm day or one aircraft's rotation never appears in both train and test. Schedule features are computed from the full published schedule, including flights later cancelled, so the model never sees day-of disruption through them.

**Model.** `HistGradientBoostingClassifier` with native categorical handling and early stopping. Training is capped at 2M rows per tier, which runs in a few minutes on a free Colab CPU.

In [ ]:
X, y, groups = modeling.make_model_frame(tat)
train_idx, test_idx = modeling.date_grouped_split(groups, test_size=0.2)
X_train, X_test, y_train, y_test = X.iloc[train_idx], X.iloc[test_idx], y[train_idx], y[test_idx]
print(f"Train: {len(X_train):,} rows on {pd.Series(groups[train_idx]).nunique()} days | "
      f"Test: {len(X_test):,} rows on {pd.Series(groups[test_idx]).nunique()} days | "
      f"base rate {y_train.mean():.1%}")

ref_idx = RNG.choice(len(X_train), size=min(300_000, len(X_train)), replace=False)
base_rate = float(y_train.mean())
models, probs, thresholds, rows = {}, {}, {}, []
for tier, features in modeling.FEATURE_TIERS.items():
    t0 = time.time()
    models[tier] = modeling.train_tier(X_train, y_train, tier, max_rows=2_000_000)
    probs[tier] = models[tier].predict_proba(X_test[features])[:, 1]
    thresholds[tier] = modeling.alert_threshold(
        models[tier].predict_proba(X_train.iloc[ref_idx][features])[:, 1], base_rate)
    flagged = probs[tier] >= thresholds[tier]
    m = modeling.evaluate(y_test, probs[tier])
    m.update(Tier=tier, Features=len(features), Iterations=models[tier].n_iter_,
             Precision_at_alert=precision_score(y_test, flagged, zero_division=0),
             Recall_at_alert=recall_score(y_test, flagged), Train_seconds=round(time.time() - t0, 1))
    rows.append(m)
    print(f"{tier:<10} ROC-AUC {m['roc_auc']:.4f} | PR-AUC {m['pr_auc']:.4f} | {m['Train_seconds']:.0f}s")

metrics_df = pd.DataFrame(rows).set_index("Tier")
display(metrics_df.round(4))
RESULTS["models"] = metrics_df.drop(columns=["Iterations"]).round(4).to_dict(orient="index")

In [ ]:
tiers = list(modeling.FEATURE_TIERS)
tier_colors = dict(zip(tiers, viz.SERIES))
fig, axes = plt.subplots(2, 2, figsize=(14.5, 11), layout="constrained")
viz.suptitle(fig, "Predicting delay-generating turnarounds: value of information")

ax = axes[0, 0]
for t in tiers:
    fpr, tpr, _ = roc_curve(y_test, probs[t])
    ax.plot(fpr, tpr, color=tier_colors[t], label=f"{t} (AUC = {metrics_df.loc[t, 'roc_auc']:.3f})")
ax.plot([0, 1], [0, 1], color=viz.NEUTRAL, ls="--", lw=1, label="Chance")
ax.set(title="ROC curve", xlabel="False positive rate", ylabel="True positive rate")
ax.legend(loc="lower right")

ax = axes[0, 1]
for t in tiers:
    prec, rec, _ = precision_recall_curve(y_test, probs[t])
    ax.plot(rec, prec, color=tier_colors[t], label=f"{t} (AP = {metrics_df.loc[t, 'pr_auc']:.3f})")
ax.axhline(y_test.mean(), color=viz.NEUTRAL, ls="--", lw=1, label=f"Base rate ({y_test.mean():.1%})")
ax.set(title="Precision-recall curve", xlabel="Recall", ylabel="Precision", ylim=(0, 1))
ax.legend(loc="upper right")

ax = axes[1, 0]
cal_max = 0.0
for t in tiers:
    frac_pos, mean_pred = calibration_curve(y_test, probs[t], n_bins=15, strategy="quantile")
    cal_max = max(cal_max, mean_pred.max(), frac_pos.max())
    ax.plot(mean_pred, frac_pos, marker="o", markersize=4, color=tier_colors[t],
            label=f"{t} (Brier = {metrics_df.loc[t, 'brier']:.3f})")
hi = cal_max * 1.1
ax.plot([0, hi], [0, hi], color=viz.NEUTRAL, ls="--", lw=1, label="Perfect calibration")
ax.set(title="Calibration (reliability diagram)", xlabel="Predicted probability", ylabel="Observed frequency")
ax.legend(loc="upper left")

ax = axes[1, 1]
aucs = metrics_df.loc[tiers, "roc_auc"]
bars = ax.bar([modeling.TIER_LABELS[t] for t in tiers], aucs - 0.5, bottom=0.5,
              color=[tier_colors[t] for t in tiers], width=0.55)
for b, v in zip(bars, aucs):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.008, f"{v:.3f}", ha="center", fontsize=11,
            fontweight="bold", color=viz.INK)
ax.set_ylim(0.5, min(1.0, aucs.max() + 0.08))
ax.set(title="ROC-AUC as the day of operation approaches", ylabel="ROC-AUC (chance = 0.5)")
ax.grid(axis="x", visible=False)

viz.save_fig(fig, "06_model_performance")
plt.show()

### 8.1 What drives the predictions?

**Permutation importance** = drop in held-out ROC-AUC when one feature is shuffled. Unlike impurity importance it is measured on test data and isn't biased toward continuous features.

In [ ]:
imp = {t: modeling.permutation_importance_table(models[t], X_test, y_test, t, n_rows=100_000, n_repeats=3)
       for t in ["Schedule", "Real-time"]}

fig, axes = plt.subplots(1, 2, figsize=(15, 5.8), layout="constrained")
viz.suptitle(fig, "Drivers of delay-generating turnarounds (permutation importance)")
for ax, t in zip(axes, ["Schedule", "Real-time"]):
    d = imp[t].sort_values("Importance")
    ax.barh(d["Feature"], d["Importance"], xerr=d["Std"], ecolor=viz.INK_2, capsize=2,
            color=[viz.ORANGE if f == "Inbound arrival delay" else viz.BLUE for f in d["Feature"]])
    ax.set(title=f"{t} tier", xlabel="Drop in ROC-AUC when shuffled")
    ax.grid(axis="y", visible=False)
viz.save_fig(fig, "07_feature_importance")
plt.show()
display(imp["Real-time"].round(4))
RESULTS["top_drivers_schedule"] = imp["Schedule"]["Feature"].head(3).tolist()
RESULTS["top_drivers_realtime"] = imp["Real-time"]["Feature"].head(3).tolist()

### 8.2 Airline risk profiles: structural vs network-driven

For each airline: the **delay-generating rate** (x) against the **AUC lift** from adding live inbound information (y = Real-time AUC − Schedule AUC, measured *within* the airline). A large lift means that airline's ground delays depend on how the inbound flight is running (network-driven). A small lift means they are already predictable from the schedule (structural).

> Why not compare *average* predicted risk between the two models? For calibrated models both averages converge to the airline's actual rate, so that gap is essentially noise. AUC lift measures what the extra information actually adds.

In [ ]:
eval_df = pd.DataFrame({"Airline": X_test["Airline"].astype(str).to_numpy(), "y": y_test,
                        "p_schedule": probs["Schedule"], "p_realtime": probs["Real-time"]})
airline_profile = modeling.auc_by_group(eval_df, "Airline", "y", ["p_schedule", "p_realtime"], min_n=2_000)
airline_profile["AUC_lift"] = airline_profile["p_realtime"] - airline_profile["p_schedule"]
display(airline_profile.sort_values("AUC_lift", ascending=False).round(4))

fig, ax = plt.subplots(figsize=(11.5, 8), layout="constrained")
x, yv = airline_profile["Delay_Rate"] * 100, airline_profile["AUC_lift"]
ax.scatter(x, yv, s=60 + 1400 * airline_profile["N"] / airline_profile["N"].max(), color=viz.BLUE,
           alpha=0.75, edgecolors="white", linewidths=1.5)
for _, r in airline_profile.iterrows():
    ax.annotate(r["Airline"], (r["Delay_Rate"] * 100, r["AUC_lift"]), xytext=(7, 5),
                textcoords="offset points", fontsize=10, fontweight="bold", color=viz.INK)
ax.margins(0.15)
ax.axvline(x.median(), color=viz.NEUTRAL, ls="--", lw=1)
ax.axhline(yv.median(), color=viz.NEUTRAL, ls="--", lw=1)
corner = dict(transform=ax.transAxes, fontsize=9, color=viz.INK_2, style="italic")
ax.text(0.02, 0.98, "Low risk · network-driven", va="top", **corner)
ax.text(0.98, 0.98, "High risk · network-driven", va="top", ha="right", **corner)
ax.text(0.02, 0.02, "Low risk · schedule-predictable", **corner)
ax.text(0.98, 0.02, "High risk · structural", ha="right", **corner)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
ax.set(title="Airline risk profiles (bubble size = turnaround volume)",
       xlabel="Delay-generating turnaround rate (held-out days)",
       ylabel="AUC lift from live inbound information")
viz.save_fig(fig, "08_airline_risk_profiles")
plt.show()

### 8.3 Surprise delays and cleared false alarms

Each model flags the riskiest turnarounds at a threshold set on **training** predictions (flag as many as the training base rate). Two outcomes show the operational value of live data:

* **Surprise delay**: the schedule-tier model said *low risk*, the real-time model said *high risk*, and the turnaround **did** generate a delay (caught only with live data).
* **Cleared false alarm**: the schedule-tier model said *high risk*, the real-time model said *low risk*, and the turnaround **did not** generate a delay.

> Both are defined against the actual outcome, not just model disagreement.

In [ ]:
flag_s = probs["Schedule"] >= thresholds["Schedule"]
flag_r = probs["Real-time"] >= thresholds["Real-time"]
eval_df["Surprise"] = (~flag_s) & flag_r & (y_test == 1)
eval_df["Cleared"] = flag_s & (~flag_r) & (y_test == 0)
print(f"Surprise delays caught only with live data : {eval_df['Surprise'].sum():,} "
      f"({eval_df['Surprise'].sum() / max(y_test.sum(), 1):.1%} of all delay-generating turnarounds)")
print(f"False alarms cleared by live data           : {eval_df['Cleared'].sum():,} "
      f"({eval_df['Cleared'].sum() / max(flag_s.sum(), 1):.1%} of schedule-tier alerts)")

by_airline = (eval_df.groupby("Airline")
              .agg(N=("y", "size"), Surprise=("Surprise", "mean"), Cleared=("Cleared", "mean"))
              .query("N >= 2000"))
fig, axes = plt.subplots(1, 2, figsize=(15, 5.8), layout="constrained")
viz.suptitle(fig, "Where live inbound information changes the call")
for ax, col, color, title in [
    (axes[0], "Surprise", viz.ORANGE, "Surprise delays (caught only with live data)"),
    (axes[1], "Cleared", viz.AQUA, "False alarms cleared by live data"),
]:
    d = by_airline[col].sort_values() * 100
    ax.barh(d.index.astype(str), d.values, color=color)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(decimals=1))
    ax.set(title=title, xlabel="Share of the airline's turnarounds")
    ax.grid(axis="y", visible=False)
viz.save_fig(fig, "09_surprise_delays")
plt.show()
RESULTS.update(surprise_share_of_delays=float(eval_df["Surprise"].sum() / max(y_test.sum(), 1)),
               cleared_share_of_alerts=float(eval_df["Cleared"].sum() / max(flag_s.sum(), 1)))

## 9. Operational Scenario Analysis

### 9.1 Risk by departure hour × weekday

Shared colour scale across airlines; cells with fewer than 30 turnarounds are left blank.

In [ ]:
top6 = tat["Airline"].value_counts().head(6).index.astype(str).tolist()
cells_ = (tat.groupby(["Airline", "HourDep", "DayOfWeek"], observed=True)["TAT_Delayed"]
             .agg(["mean", "size"]).reset_index())
cells_["Airline"] = cells_["Airline"].astype(str)
cells_ = cells_[cells_["size"] >= 30]
vmin, vmax = (cells_.loc[cells_["Airline"].isin(top6), "mean"].quantile([0.02, 0.98]) * 100).tolist()

fig, axes = plt.subplots(2, 3, figsize=(17, 12), layout="constrained")
viz.suptitle(fig, "Delay-generating turnaround rate by scheduled departure hour and weekday")
for i, (ax, airline) in enumerate(zip(axes.flat, top6)):
    h = (cells_[cells_["Airline"] == airline]
         .pivot(index="HourDep", columns="DayOfWeek", values="mean")
         .reindex(index=range(24), columns=range(7)) * 100)
    sns.heatmap(h, ax=ax, cmap=viz.SEQ_CMAP, vmin=vmin, vmax=vmax, cbar=False, linewidths=0.4,
                linecolor="white", xticklabels=cfg.DAY_LABELS,
                yticklabels=[f"{hh:02d}:00" for hh in range(24)] if i % 3 == 0 else False)
    ax.set_facecolor("#f0efec")
    ax.grid(False)
    ax.set_title(f"Airline {airline}", loc="left")
    ax.set_xlabel("")
    ax.set_ylabel("Scheduled departure" if i % 3 == 0 else "")
    ax.tick_params(axis="y", labelsize=7)
fig.colorbar(ScalarMappable(Normalize(vmin, vmax), viz.SEQ_CMAP), ax=axes, shrink=0.5,
             label="Delay-generating turnarounds (%)")
viz.save_fig(fig, "10_hour_weekday_heatmap")
plt.show()

overall = (tat.groupby(["DayOfWeek", "HourDep"])["TAT_Delayed"].agg(["mean", "size"])
              .query("size >= 500").reset_index())
worst, best = overall.loc[overall["mean"].idxmax()], overall.loc[overall["mean"].idxmin()]
fmt = lambda r: f"{cfg.DAY_LABELS[int(r['DayOfWeek'])]} {int(r['HourDep']):02d}:00 ({r['mean']:.1%}, n={int(r['size']):,})"
print("Riskiest departure window:", fmt(worst))
print("Safest departure window  :", fmt(best))
RESULTS.update(riskiest_window=fmt(worst), safest_window=fmt(best))

### 9.2 Seasonal risk calendar by airline

In [ ]:
season = (tat.groupby(["Airline", "Month"], observed=True)["TAT_Delayed"].agg(["mean", "size"]).reset_index())
season["Airline"] = season["Airline"].astype(str)
season = season[season["size"] >= 100]
season_pivot = season.pivot(index="Airline", columns="Month", values="mean").reindex(columns=range(1, 13))
season_pivot.columns = cfg.MONTH_LABELS  # safe: columns were reindexed to exactly months 1..12
season_pivot = season_pivot.loc[season_pivot.mean(axis=1).sort_values(ascending=False).index]
monthly_overall = tat.groupby("Month")["TAT_Delayed"].mean().reindex(range(1, 13))
peak_month, low_month = int(monthly_overall.idxmax()), int(monthly_overall.idxmin())

fig, axes = plt.subplots(1, 2, figsize=(18, 7), layout="constrained", gridspec_kw={"width_ratios": [1.15, 1]})
viz.suptitle(fig, "Seasonal turnaround risk calendar by airline")
sns.heatmap(season_pivot * 100, ax=axes[0], cmap=viz.SEQ_CMAP, annot=True, fmt=".0f", linewidths=0.5,
            linecolor="white", annot_kws={"fontsize": 8}, cbar_kws={"label": "Delay-generating turnarounds (%)"})
axes[0].set_facecolor("#f0efec")
axes[0].grid(False)
axes[0].set(title="Rate by airline and month (%)", xlabel="", ylabel="")
axes[0].tick_params(axis="y", rotation=0)

ax = axes[1]
ax.axvspan(peak_month - 0.5, peak_month + 0.5, color=viz.GRID, alpha=0.8, lw=0)
for airline, color in zip(top6, viz.SERIES):
    if airline in season_pivot.index:
        ax.plot(range(1, 13), season_pivot.loc[airline].values * 100, marker="o", markersize=4,
                color=color, label=airline)
ax.set_xticks(range(1, 13), cfg.MONTH_LABELS)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
ax.set(title=f"Six largest airlines (shaded: network peak, {cfg.MONTH_LABELS[peak_month - 1]})",
       ylabel="Delay-generating turnarounds")
ax.legend(ncol=3, fontsize=9)
viz.save_fig(fig, "11_seasonal_calendar_airlines")
plt.show()
print(f"Highest-risk month: {cfg.MONTH_LABELS[peak_month - 1]} ({monthly_overall.max():.1%}) | "
      f"lowest: {cfg.MONTH_LABELS[low_month - 1]} ({monthly_overall.min():.1%})")
RESULTS.update(peak_month=cfg.MONTH_LABELS[peak_month - 1], peak_month_rate=float(monthly_overall.max()),
               low_month=cfg.MONTH_LABELS[low_month - 1], low_month_rate=float(monthly_overall.min()))

## 10. Network Risk Analysis

### 10.1 Interactive US risk map

GitHub's notebook viewer strips JavaScript, so an interactive Plotly chart shows up there as an empty cell. Two copies are saved instead:

* `docs/index.html` is the fully interactive map. Enable **GitHub Pages → /docs** to get a live link.
* `images/12_us_risk_map.png` is a static copy for the README.

In [ ]:
coords = geo.load_airport_coords()
airport_map = geo.airport_risk_table(tat, coords, min_events=300)
outside = ~(airport_map["Longitude"].between(-126, -66) & airport_map["Latitude"].between(24, 50))
print(f"{len(airport_map)} airports mapped, {outside.sum()} of them in Alaska, Hawaii or US territories")

risk_fig = geo.risk_map_figure(airport_map, cfg.YEAR)
risk_fig.write_html(cfg.DOCS_DIR / "index.html", include_plotlyjs="cdn", full_html=True)
how = geo.export_static_map(risk_fig, cfg.IMAGES_DIR / "12_us_risk_map.png", airport_map)
print(f"Saved docs/index.html and images/12_us_risk_map.png (static export via {how})")
risk_fig.show()

top_risk = (airport_map.query("Events >= 1000").nlargest(10, "Delay_Pct")
            [["Origin", "City", "Delay_Pct", "Mean_TAT", "Median_Sched_TAT", "Events"]])
display(top_risk.reset_index(drop=True))
RESULTS["riskiest_airports"] = top_risk["Origin"].head(5).tolist()

### 10.2 Turn-buffer reliability

*If an airline schedules an X-minute turn at this airport, how often does it hold?* Reliability is the share of turnarounds that did **not** generate a delay, broken down by **planned** ground time.

> Why not simply report P(actual ground time ≤ X)? That mostly reflects how long airlines *plan* to stay at each airport (hubs are scheduled with long turns), not how reliable the turnarounds are. Conditioning on the planned buffer removes that confound.

In [ ]:
buffer_labels = ["≤45 min", "46–60 min", "61–90 min", "91–120 min", ">120 min"]
tat_buf = tat[["Origin", "TAT_Scheduled", "TAT_Delayed"]].assign(
    Buffer=pd.cut(tat["TAT_Scheduled"], bins=[0, 45, 60, 90, 120, cfg.MAX_SCHED_TAT], labels=buffer_labels))
top25 = tat["Origin"].value_counts().head(25).index
rel = (tat_buf[tat_buf["Origin"].isin(top25)]
       .groupby(["Origin", "Buffer"], observed=True)["TAT_Delayed"].agg(["mean", "size"]).reset_index())
rel["Origin"] = rel["Origin"].astype(str)
rel["Reliability"] = np.where(rel["size"] >= 200, (1 - rel["mean"]) * 100, np.nan)
reliability = rel.pivot(index="Origin", columns="Buffer", values="Reliability").reindex(columns=buffer_labels)
reliability = reliability.loc[reliability.mean(axis=1).sort_values().index]

fig, ax = plt.subplots(figsize=(11, 11), layout="constrained")
sns.heatmap(reliability, ax=ax, cmap=viz.SEQ_CMAP, annot=True, fmt=".0f", linewidths=0.5, linecolor="white",
            cbar_kws={"label": "Turnarounds completed without generating a delay (%)", "shrink": 0.6})
ax.set_facecolor("#f0efec")
ax.grid(False)
ax.set_title("Turn-buffer reliability at the 25 busiest airports\n(blank = fewer than 200 turnarounds)",
             loc="left", fontsize=13, fontweight="bold")
ax.set(xlabel="Planned ground time", ylabel="Airport (least reliable first)")
ax.tick_params(axis="y", rotation=0)
viz.save_fig(fig, "13_turn_buffer_reliability")
plt.show()

share = (pd.crosstab(tat_buf["Origin"], tat_buf["Buffer"], normalize="index") * 100).reindex(top25.astype(str)).round(1)
print("Mix of planned ground times at the 10 busiest airports (% of turnarounds):")
display(share.head(10))

### 10.3 Airport seasonal risk calendar

In [ ]:
top20 = tat["Origin"].value_counts().head(20).index
ap = (tat[tat["Origin"].isin(top20)].groupby(["Origin", "Month"], observed=True)["TAT_Delayed"]
      .agg(["mean", "size"]).reset_index())
ap["Origin"] = ap["Origin"].astype(str)
ap = ap[ap["size"] >= 50]
ap_pivot = ap.pivot(index="Origin", columns="Month", values="mean").reindex(columns=range(1, 13))
ap_pivot.columns = cfg.MONTH_LABELS
summer = ap_pivot[["Jun", "Jul", "Aug"]].mean(axis=1)
ap_pivot = ap_pivot.loc[summer.sort_values(ascending=False).index]

fig, axes = plt.subplots(1, 2, figsize=(19, 8), layout="constrained", gridspec_kw={"width_ratios": [1.2, 1]})
viz.suptitle(fig, "Seasonal turnaround risk at the 20 busiest US airports")
sns.heatmap(ap_pivot * 100, ax=axes[0], cmap=viz.SEQ_CMAP, annot=True, fmt=".0f", linewidths=0.4,
            linecolor="white", annot_kws={"fontsize": 8}, cbar_kws={"label": "Delay-generating turnarounds (%)"})
axes[0].set_facecolor("#f0efec")
axes[0].grid(False)
axes[0].set(title="Rate by airport and month (%), sorted by summer (Jun–Aug) risk", xlabel="", ylabel="")
axes[0].tick_params(axis="y", rotation=0)

ax = axes[1]
for airport, color in zip(ap_pivot.index[:5], viz.SERIES):
    ax.plot(range(1, 13), ap_pivot.loc[airport].values * 100, marker="o", markersize=4, color=color, label=airport)
ax.set_xticks(range(1, 13), cfg.MONTH_LABELS)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
ax.set(title="Five airports with the highest summer risk", ylabel="Delay-generating turnarounds")
ax.legend()
viz.save_fig(fig, "14_airport_seasonal_calendar")
plt.show()
print("Highest summer (Jun–Aug) risk:")
print((summer.sort_values(ascending=False).head(5) * 100).round(1).to_string())
print("\nHighest December risk:")
print((ap_pivot["Dec"].nlargest(5) * 100).round(1).to_string())

## 11. Key Results & Export

Writes `outputs/metrics.json`, `outputs/key_results.md` (paste-ready for the README) and the result tables as CSV. On Colab, the final cell downloads `images/`, `docs/` and `outputs/` as one zip so you can copy them into your local repository.

In [ ]:
def _jsonable(o):
    return o.item() if hasattr(o, "item") else str(o)

(cfg.OUTPUTS_DIR / "metrics.json").write_text(json.dumps(RESULTS, indent=2, default=_jsonable))
tables = {
    "turnaround_funnel": pd.DataFrame(funnel, columns=["Step", "Rows"]),
    "distribution_fit": fit_table,
    "entropy_by_airline": entropy_airline,
    "uncertainty_mean_tat": unc_tat,
    "uncertainty_delay_rate": unc_rate,
    "airport_clusters": feats.reset_index(),
    "model_metrics": metrics_df.reset_index(),
    "feature_importance_realtime": imp["Real-time"],
    "airline_risk_profiles": airline_profile,
    "airport_risk": airport_map,
    "turn_buffer_reliability": reliability.reset_index(),
}
for name, t in tables.items():
    t.to_csv(cfg.OUTPUTS_DIR / f"{name}.csv", index=False)

m = RESULTS["models"]
key_results = f'''| Metric | Value |
|---|---|
| Flights analysed | {RESULTS['flights_raw']:,} |
| Turnarounds reconstructed | {RESULTS['turnarounds']:,} ({RESULTS['aircraft']:,} aircraft, {RESULTS['airlines']} airlines, {RESULTS['airports']} airports) |
| Median actual / planned ground time | {RESULTS['median_tat']:.0f} min / {RESULTS['median_sched_tat']:.0f} min |
| Delay-generating turnarounds | {RESULTS['delayed_rate']:.1%} (naive 'actual > planned' label: {RESULTS['naive_delayed_rate']:.1%}) |
| Multi-day gaps a naive clock-modulo method mistakes for turnarounds | {RESULTS['naive_aliased']:,} |
| Best-fitting distribution | {RESULTS['best_distribution']} (KS = {RESULTS['best_ks']:.3f}) |
| KL divergence, hubs vs regional | {RESULTS['kl_hub_regional']:.2f} bits (JS distance {RESULTS['js_hub_regional']:.2f}) |
| Median design effect (aircraft clustering) | {RESULTS['design_effect_tat_median']:.1f}× for mean ground time, {RESULTS['design_effect_rate_median']:.1f}× for delay rate |
| ROC-AUC: calendar → schedule → real-time | {m['Calendar']['roc_auc']:.3f} → {m['Schedule']['roc_auc']:.3f} → {m['Real-time']['roc_auc']:.3f} |
| PR-AUC: calendar → schedule → real-time | {m['Calendar']['pr_auc']:.3f} → {m['Schedule']['pr_auc']:.3f} → {m['Real-time']['pr_auc']:.3f} |
| Delays caught only with live inbound data | {RESULTS['surprise_share_of_delays']:.1%} of delay-generating turnarounds |
| Riskiest / safest departure window | {RESULTS['riskiest_window']} / {RESULTS['safest_window']} |
| Highest / lowest risk month | {RESULTS['peak_month']} ({RESULTS['peak_month_rate']:.1%}) / {RESULTS['low_month']} ({RESULTS['low_month_rate']:.1%}) |
| Highest-risk large airports | {', '.join(RESULTS['riskiest_airports'])} |
'''
(cfg.OUTPUTS_DIR / "key_results.md").write_text(key_results)
print(key_results)

In [ ]:
# Colab only: bundle the generated artefacts for download.
if IN_COLAB:
    import zipfile
    from google.colab import files

    bundle = Path("/content/turnaround_results.zip")
    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
        for folder in ("images", "docs", "outputs"):
            for f in (ROOT / folder).rglob("*"):
                if f.is_file():
                    zf.write(f, f.relative_to(ROOT))
    files.download(str(bundle))
else:
    print("Artefacts written to images/, docs/ and outputs/.")